# Solutions to Exercises 12: More SPARQL

1. Load in the ontology `teaching.rdf` and 

a) Write a query using the OWLReady2 search capability to find all entities to which the entity with attribute `program_title="MSc Artificial Intelligence` has a direct relation (in either direction). You will need to use the ontology to help you with this. Plot the resulting network.


A `Program` can be involved in the following relations:

* `offers_programme(School, Program)`
* `has_module(Program, Module)`
* `is_enrolled_on(Student,Program)`
* `is_directed_by(Program,Staff)`

The other relation present in the ontology is 

* `is_taught_by(module,staff)`

In [20]:
from owlready2 import *
onto = get_ontology("teaching.rdf").load()
program = onto.search_one(program_title="MSc Artificial Intelligence")
students = onto.search(is_enrolled_on = program)
director = program.is_directed_by
modules = program.has_module
school = onto.search(offers_programme = program)
print(students)
print(director)
print(modules)
print(school)

[teaching.stu01, teaching.stu02, teaching.stu03, teaching.stu04, teaching.stu05]
[teaching.sta01]
[teaching.mod01, teaching.mod02, teaching.mod03, teaching.mod04, teaching.mod05, teaching.mod06, teaching.mod07]
[teaching.sch01]


Form these into triples for visualisation

In [29]:
ntriples = []

for i in students:
    ntriples.append((i.name, 'is_enrolled_on', program.name))


for i in director:
    ntriples.append((i.name, 'is_directed_by', program.name))


for i in modules:
    ntriples.append((program.name, 'has_module', i.name))

for i in school:
    ntriples.append((i.name, 'offers_programme', program.name))



Now get relations between the nodes: the only relation that allows this is `is_taught_by`

In [30]:
for d in director:
    teaches = onto.search(is_taught_by=d)
    for i in teaches:
        if i in modules:
            ntriples.append((i.name, 'is_taught_by', d.name))

Now we can form the graph

In [35]:
from pyvis.network import Network
net = Network()
# Get the node names
nodenames = set()
for i in ntriples:
    nodenames.add(i[0])
    nodenames.add(i[2])

for n in nodenames:
    net.add_node(n)

for n in ntriples:
    net.add_edge(n[0], n[2], title=n[1])

net.toggle_physics(True)
net.repulsion()
net.show_buttons(filter_=['physics'])
net.save_graph("nx.html")


    b) Repeat part (a) using SPARQL.

In [38]:
outgoing = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT $program ?relation ?entity
    WHERE
    {
        ?program rdf:type ONTO:Program
        ?program ONTO:program_title ?title
        $program ?relation ?entity
        FILTER($title="MSc Artificial Intelligence")
        FILTER(STRSTARTS(STR(?relation), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?entity), "http://www.dummy.info/new.owl#"))

    }
    """))

for i in outgoing:  
    print(i)

[teaching.pro01, teaching.has_module, teaching.mod01]
[teaching.pro01, teaching.has_module, teaching.mod02]
[teaching.pro01, teaching.has_module, teaching.mod03]
[teaching.pro01, teaching.has_module, teaching.mod04]
[teaching.pro01, teaching.has_module, teaching.mod05]
[teaching.pro01, teaching.has_module, teaching.mod06]
[teaching.pro01, teaching.has_module, teaching.mod07]
[teaching.pro01, teaching.is_directed_by, teaching.sta01]


In [40]:
incoming = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT $entity ?relation ?program
    WHERE
    {
        ?program rdf:type ONTO:Program
        ?program ONTO:program_title ?title
        $entity ?relation ?program
        FILTER($title="MSc Artificial Intelligence")
        FILTER(STRSTARTS(STR(?relation), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?entity), "http://www.dummy.info/new.owl#"))

    }
    """))

for i in incoming:  
    print(i)

[teaching.sch01, teaching.offers_programme, teaching.pro01]
[teaching.stu01, teaching.is_enrolled_on, teaching.pro01]
[teaching.stu02, teaching.is_enrolled_on, teaching.pro01]
[teaching.stu03, teaching.is_enrolled_on, teaching.pro01]
[teaching.stu04, teaching.is_enrolled_on, teaching.pro01]
[teaching.stu05, teaching.is_enrolled_on, teaching.pro01]


Now get the cross-links. This is turns out is really hard to do. Here's an examplem of how to do it. Note that there are several permutations of this because of the different directions of the three relations involved.

In [57]:
outgoingx = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT $entity ?relation2 ?entity2
    WHERE
    {
        ?program rdf:type ONTO:Program
        ?program ONTO:program_title ?title
        $program ?relation ?entity
        $program ?relation2 ?entity2
        ?entity ?relation3 ?entity2
        FILTER($title="MSc Artificial Intelligence")
        FILTER(STRSTARTS(STR(?relation), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?entity), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?relation2), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?entity2), "http://www.dummy.info/new.owl#"))
        FILTER(STRSTARTS(STR(?relation3), "http://www.dummy.info/new.owl#"))
    }
    """))

for i in outgoingx:  
    print(i)

[teaching.mod05, teaching.is_directed_by, teaching.sta01]


2. Load in the ontology `HALD.rdf`.

a) Write a query to extract the 1- and 2- neighbourhoods of the `Disease` entity "Prostatic Neoplasms" and plot the graph. You may use  SPARQL or OWLReady2's `search` function for this.


In [7]:
# Placeholder for code

b) By systematically growing size of the neighbourhood, find a pathway from "Prostatic Neoplasms" to "Prostatis" (hint: based on the names, do you expect these to be near or far apart?)

In [8]:
# Placeholder for code